# ECG Experiment Notebook: T28_high_dropout

- **Architecture:** transformer
- **Preprocessing/Filter:** bandpass_notch
- **Balancing Mode:** none
- **Loss Function:** asl


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from data_management.dataset_factory import DatasetFactory
from mastermind_loop import build_model, build_preprocessing_pipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

## 1. Load Preprocessing and Model Checkpoint

In [ ]:
preprocessor, _ = build_preprocessing_pipeline('bandpass_notch')
model = build_model('transformer').to(device)
checkpoint_path = 'models/T28_high_dropout_best.pt'
try:
    model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=False))
    print('Successfully loaded checkpoint from', checkpoint_path)
except Exception as e:
    print('Error loading checkpoint:', e)
model.eval()

## 2. Generate and Analyze Embeddings (UMAP, t-SNE, Silhouette score)

In [ ]:
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
train_ds, val_ds, test_ds, _ = DatasetFactory.create_datasets(
    dataset_type='ptbxl', download=False, resolution='lr', preprocessor=preprocessor
)
embeddings, targets = [], []
for i in range(min(500, len(test_ds))):
    x, y = test_ds[i]
    x_t = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        z = model.get_representation(x_t).cpu().numpy()
    embeddings.append(z[0])
    targets.append(y)

embeddings = np.array(embeddings)
targets = np.array(targets)
print('Embeddings shape:', embeddings.shape)

tsne = TSNE(n_components=2, random_state=42)
proj = tsne.fit_transform(embeddings)
plt.figure(figsize=(8, 6))
plt.scatter(proj[:, 0], proj[:, 1], c=targets.argmax(axis=1), cmap='tab10', alpha=0.7)
plt.colorbar(label='Dominant Class Index')
plt.title(f't-SNE Embeddings ({trial_id})')
plt.show()

try:
    sil = silhouette_score(embeddings, targets.argmax(axis=1))
    print('Silhouette Score:', sil)
except Exception as e:
    print('Could not compute Silhouette:', e)